In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

ratings.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
items = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

items = items[[0, 1]]
items.columns = ["item_id", "title"]

items.head()

,item_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [6]:
item_id_to_title = dict(zip(items["item_id"], items["title"]))

In [7]:
item_id_to_title[1]

'Toy Story (1995)'

In [8]:
user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating"
)

In [9]:
user_item_matrix.shape

(943, 1682)

In [10]:
user_item_matrix_filled = user_item_matrix.fillna(0)

In [11]:
user_item_matrix_filled.iloc[:5, :5]

item_id,1,2,3,4,5
user_id,,,,,
1,5.0,3.0,4.0,3.0,3.0
2,4.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0


In [12]:
item_similarity = cosine_similarity(user_item_matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

In [13]:
item_similarity_df.shape

(1682, 1682)

In [14]:
def recommend_items(item_id, num_recommendations=5):
    if item_id not in item_similarity_df.columns:
        return "Item not found"

    similar_scores = item_similarity_df[item_id].sort_values(ascending=False)
    return similar_scores.iloc[1:num_recommendations+1]

In [15]:
recommend_items(1)

item_id
50     0.734572
181    0.699925
121    0.689786
117    0.664555
405    0.641322
Name: 1, dtype: float64

In [16]:
items_full = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

item_features = items_full.iloc[:, 5:]

In [17]:
item_features.shape

(1682, 19)

In [18]:
content_similarity = cosine_similarity(item_features)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=items_full[0],
    columns=items_full[0]
)

In [19]:
content_similarity_df.shape

(1682, 1682)

In [20]:
def hybrid_recommendation(item_id, alpha=0.5, num_recommendations=5):
    if item_id not in item_similarity_df.columns:
        return "Item not found"

    cf_scores = item_similarity_df[item_id]
    cb_scores = content_similarity_df[item_id]

    hybrid_scores = alpha * cf_scores + (1 - alpha) * cb_scores
    hybrid_scores = hybrid_scores.sort_values(ascending=False)

    return hybrid_scores.iloc[1:num_recommendations+1]

In [21]:
hybrid_recommendation(1)

item_id
95     0.723999
151    0.652412
240    0.623647
94     0.614189
225    0.612941
Name: 1, dtype: float64

In [22]:
def hybrid_with_names(item_id, alpha=0.5, num_recommendations=5):
    scores = hybrid_recommendation(item_id, alpha, num_recommendations)

    results = []
    for i in scores.index:
        title = item_id_to_title.get(i, "Unknown")
        score = round(float(scores[i]), 3)  # ✅ clean + round
        results.append((title, score))

    return results

In [23]:
hybrid_with_names(1)

[('Aladdin (1992)', 0.724),
 ('Willy Wonka and the Chocolate Factory (1971)', 0.652),
 ('Beavis and Butt-head Do America (1996)', 0.624),
 ('Home Alone (1990)', 0.614),
 ('101 Dalmatians (1996)', 0.613)]

In [100]:
test_users = user_item_matrix.index[:50]

In [101]:
def precision_at_k(k=5):
    precisions = []

    for user in test_users:
        user_ratings = user_item_matrix.loc[user].dropna()

        liked_items = user_ratings[user_ratings >= 4].index
        if len(liked_items) == 0:
            continue

        test_item = liked_items[0]

        recs = hybrid_recommendation(test_item, num_recommendations=k)

        relevant = 0
        for item in recs.index:
            if item in liked_items:
                relevant += 1

        precisions.append(relevant / k)

    return sum(precisions) / len(precisions)

In [102]:
precision_at_k(5)

0.20800000000000002